In [1]:
import os
import gc
import copy
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import timm 
from torch import nn
from torch.optim import Adam
from torchvision import models
import torch.nn.functional as F
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchinfo import summary
from typing import List, Tuple, Union
from PIL import Image
import subprocess
import platform
import psutil

In [2]:
# Hyperparameters

BATCH_SIZE = 32
EPOCHS = 100
NUM_CLASSES = 4
DROPOUT_RATE = 0.3
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

In [3]:
# Model name
MODEL1_NAME = 'DenseNet121'
MODEL2_NAME = 'MobileNetV2'
MODEL3_NAME = 'ResNet50'
MODEL4_NAME = 'VGG19'

In [4]:
# Train directory path

TRAIN_DIR = '/kaggle/input/brain-tumor-mri-dataset/Training'
TRAIN_DIR

'/kaggle/input/brain-tumor-mri-dataset/Training'

In [5]:
TEST_DIR = '/kaggle/input/brain-tumor-mri-dataset/Testing'

In [6]:
# Resnet50
MODEL3_DIR = '/kaggle/input/datasets/hophamsailam/resnet50-kaggle-train-test-validation/ResNet50.pth'
# VGG16
MODEL4_DIR = '/kaggle/input/datasets/hophamsailam/vgg19-kaggle-train-test-validation/VGG19.pth'

### Student 

# EdgeneXt-XXS
MODEL5_DIR = '/kaggle/input/datasets/xvmhieu/kaggle-edgenextxxs-ben-113/student_distilled.pth'
# MobileVit-XXS
MODEL6_DIR = '/kaggle/input/datasets/xvmhieu/kaggle-mobilevitxxs-beb-113/student_distilled.pth'

In [7]:
# 4 labels: Glioma Tumor, Meningioma Tumor, Pituitary Tumor, No Tumor

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
CLASS_NAMES

['glioma', 'meningioma', 'notumor', 'pituitary']

In [8]:
# Enable cuDNN benchmark for optimal performance during inference profiling
SEED = 24520152

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

## DEVICE
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    cpu_info = subprocess.check_output("lscpu", shell=True).decode('utf-8')
    print(cpu_info)
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Running on GPU:{gpu_name}")

print(f"Inference profiling initialized on device: {DEVICE.upper()}")

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  4
On-line CPU(s) list:                     0-3
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   79
Thread(s) per core:                      2
Core(s) per socket:                      2
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4399.99
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq 

In [9]:
def get_data_loaders(train_dir: str = TRAIN_DIR, test_dir: str = TEST_DIR, batch_size: int = BATCH_SIZE) -> tuple[DataLoader, DataLoader]:
    """
    Creates PyTorch DataLoaders from train and test directories.

    Args:
        train_dir (str): Path to the training dataset directory.
        test_dir (str): Path to the test/validation dataset directory.
        batch_size (int): Batch size.

    Returns:
        tuple[DataLoader, DataLoader]: (train_loader, val_loader)
    """
    
    # Standard ImageNet normalization statistics
    norm_mean=[0.485, 0.456, 0.406]
    norm_std=[0.229, 0.224, 0.225]

    # Training Transform Pipeline
    train_transform = v2.Compose([
        # Resize to 256x256 first. This provides a buffer for subsequent 
        # rotation/translation and cropping, preventing black border artifacts.
        v2.Resize(size=256),

        # Apply Data Augmentation
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(degrees=36),
        v2.RandomAffine(degrees=0, scale=(0.9, 1.1)),
        v2.ColorJitter(brightness=0.1, contrast=0.1),

        # Use CenterCrop to focus on the primary subject
        v2.CenterCrop(size=224),

        # Convert PIL/Numpy to Tensor, cast to Float32, and rescale to [0, 1]
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),

        # Normalize using ImageNet mean and std
        v2.Normalize(mean=norm_mean, std=norm_std),
    ])

    # Validation Transform Pipeline
    val_transform = v2.Compose([
        v2.Resize(size=256),
        v2.CenterCrop(size=224),
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=norm_mean, std=norm_std)
    ])

    # Worker Configuration
    # Determine the optimal number of CPU workers to prevent bottlenecks.
    # Capped at 4 to avoid excessive memory overhead.
    num_workers = min(4, os.cpu_count())

    train_dataset = ImageFolder(root=train_dir, transform=train_transform)
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)

    val_dataset = ImageFolder(root=test_dir, transform=val_transform)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader

In [10]:
class DenseNet121(nn.Module):
    """
    DenseNet121-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: DenseNet121 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()

        # Load Pre-trained DenseNet121
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        backbone = models.densenet121(weights=weights)

        # DenseNet121 .features contains all Conv/Relu/MaxPool layers
        self.features = backbone.features

        # Unfreezing parameters for base evaluation. 
        for param in self.features.parameters():
            param.requires_grad = True  

        # Define Custom Classifier Head.
        self.in_features = 1024 
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 1024, H, W) -> (Batch, 1024, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1024, 1, 1) -> (Batch, 1024)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W).
                              Expected standard ImageNet normalization.

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """
        
        # Feature extraction (Frozen)
        x = self.features(x)
        # The torchvision.models.densenet121 `.features` block ends with a 
        # BatchNorm layer (norm5), which outputs both negative and positive values.
        # We MUST apply ReLU here to zero out negative values (noise/background).
        x = F.relu(x, inplace=True)
        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        
        # Output Logits
        logits = self.classifier(x)
        
        return logits

In [11]:
def build_densenet121(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> DenseNet121:
    """
    Factory function to instantiate the customized DenseNet121 model for Transfer Learning.

    This function initializes a `DenseNet121` which includes:
    1. A frozen DenseNet121 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (ReLU -> Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        DenseNet121: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """
    
    model = DenseNet121(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [12]:
class MobileNetV2(nn.Module):
    """
    MobileNetV2-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: MobileNetV2 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> None:
        super().__init__()

        # Load Pre-trained MobileNetV2
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        backbone = models.mobilenet_v2(weights=weights)

        # MobileNetV2 .features contains all the convolutional layers (Inverted Residuals)
        self.features = backbone.features

        # Unfreeze ALL
        for param in self.features.parameters():
            param.requires_grad = True

        # Define Custom Classifier Head
        # MobileNetV2 output feature map has 1280 channels
        self.in_features = 1280 
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 1280, H, W) -> (Batch, 1280, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1280, 1, 1) -> (Batch, 1280)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [13]:
def build_mobilenetv2(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> MobileNetV2:
    """
    Factory function to instantiate the customized MobileNetV2 model for Transfer Learning.

    This function initializes a `MobileNetV2` which includes:
    1. A frozen MobileNetV2 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        MobileNetV2: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = MobileNetV2(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [14]:
class ResNet50(nn.Module):
    """
    ResNet50-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: ResNet50 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()

        # Load Pre-trained ResNet50
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        original_model = models.resnet50(weights=weights)

        # Feature Extractor
        # ResNet50 structure: [conv1, bn1, ..., layer1, layer2, layer3, layer4, avgpool, fc]
        # We remove the last 2 layers ('avgpool' and 'fc') to keep only the convolutional part.
        self.features = nn.Sequential(*list(original_model.children())[:-2])

        # Unfreeze ALL
        for param in self.features.parameters():
            param.requires_grad = True

        # 4. Define Custom Classifier Head
        # ResNet50's final conv block (layer4) outputs 2048 channels.
        self.in_features = 2048 
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 2048, H, W) -> (Batch, 2048, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 2048, 1, 1) -> (Batch, 2048)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [15]:
def build_resnet50(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> ResNet50:
    """
    Factory function to instantiate the customized ResNet50 model for Transfer Learning.

    This function initializes a `ResNet50` which includes:
    1. A frozen ResNet50 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        ResNet50: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = ResNet50(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [16]:
class VGG19(nn.Module):
    """
    VGG19-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: VGG19 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> None:
        super().__init__()

        # Load Pre-trained VGG19
        weights = models.VGG19_Weights.IMAGENET1K_V1
        backbone = models.vgg19(weights=weights)

        # VGG19 .features contains all Conv/Relu/MaxPool layers
        self.features = backbone.features

        # Unfreeze ALL
        for param in self.features.parameters():
            param.requires_grad = True

        # Define Custom Classifier Head
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 512, H, W) -> (Batch, 512, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 512, 1, 1) -> (Batch, 512)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=512, out_features=num_classes) # VGG19 features output exactly 512 channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [17]:
def build_vgg19(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> VGG19:
    """
    Factory function to instantiate the customized VGG19 model for Transfer Learning.

    This function initializes a `VGG19` which includes:
    1. A frozen VGG19 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        VGG19Classifier: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = VGG19(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [18]:
def build_edgenext_xxs_model(num_classes: int = 4) -> nn.Module:
    """Builds a lightweight Student Model (EdgeNeXt-XXS) for distillation.
    
    This function instantiates the Extra-Extra-Small variant of EdgeNeXt via `timm`. 
    Introduced in ECCV 2022, EdgeNeXt is a state-of-the-art hybrid architecture 
    that amalgamates CNNs and Vision Transformers. It employs Split Depth-wise 
    Transposed Attention (SDTA) to effectively capture global context while 
    minimizing the computational overhead typically associated with ViTs.

    Args:
        num_classes (int, optional): Number of output classes for the 
            classification head. Defaults to 4.

    Returns:
        nn.Module: The initialized EdgeNeXt-XXS PyTorch model (~1.3M params).
    """
    print("Initializing EdgeNeXt-XXS student model...")
    
    # timm seamlessly integrates pre-trained weights and adapts the classifier
    model = timm.create_model(
        model_name='edgenext_xx_small', 
        pretrained=True, 
        num_classes=num_classes
    )
    
    return model

In [19]:
def build_mobilevit_xxs_model(num_classes: int = 4) -> nn.Module:
    """Builds a lightweight Student Model (MobileViT-XXS) for distillation.
    
    This function instantiates the Extra-Extra-Small (XXS) variant of MobileViT 
    using the `timm` library. MobileViT is a hybrid architecture that seamlessly 
    combines the spatial inductive biases of Convolutional Neural Networks (CNNs) 
    with the global attention mechanisms of Vision Transformers (ViTs).

    Args:
        num_classes (int, optional): Number of output classes for the 
            classification head. Defaults to 4.

    Returns:
        nn.Module: The initialized MobileViT-XXS PyTorch model (~1.2M params).
    """
    print("Initializing MobileViT-XXS student model...")
    
    # The timm library automatically downloads the pre-trained ImageNet weights 
    # and safely replaces the final classification head to match `num_classes`.
    model = timm.create_model(
        model_name='mobilevit_xxs', 
        pretrained=True, 
        num_classes=num_classes
    )
    
    return model

In [20]:
def get_tta_transform() -> v2.Compose:
    """Constructs the Test-Time Augmentation (TTA) pipeline.
    
    Returns:
        v2.Compose: A composition of torchvision transforms.
    """
    return v2.Compose([
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.1, contrast=(0.9, 1.1))
    ])


class HeavyTeacherPipeline:
    """Simulates the inference pipeline of the Heavy Teacher ensemble with TTA.
    
    Attributes:
        models (List[nn.Module]): List of loaded PyTorch models.
        tta_rounds (int): Number of Test-Time Augmentation iterations.
        tta_transform (v2.Compose): The augmentation transformations.
        device (str): Computation device ('cuda' or 'cpu').
    """
    
    def __init__(self, models_list: List[nn.Module], tta_rounds: int = 5, num_classes: int = NUM_CLASSES, device: str = DEVICE) -> None:
        """Initializes the pipeline with the given models and TTA configuration."""
        self.models = [model.to(device).eval() for model in models_list]
        self.tta_rounds = tta_rounds
        self.tta_transform = get_tta_transform()
        self.num_classes = num_classes
        self.device = device
        self.norm = v2.Normalize(mean=NORM_MEAN, std=NORM_STD)

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        """Performs ensemble inference with TTA on a single input tensor."""
        # x is Normalized [1, 3, 224, 224]
        final_probs = torch.zeros((x.size(0), self.num_classes), device=self.device)
        
        #  Un-Normalize (multiply std, plus mean)
        mean_tensor = torch.tensor(NORM_MEAN, device=self.device).view(1, 3, 1, 1)
        std_tensor = torch.tensor(NORM_STD, device=self.device).view(1, 3, 1, 1)
        
        with torch.no_grad():
            for model in self.models:
                model_probs = torch.zeros((x.size(0), self.num_classes), device=self.device)
                
                # 1. Predict
                logits = model(x)
                model_probs += torch.softmax(logits, dim=1)
                
                # 2. predict on TTA
                if self.tta_rounds > 0:
                    # Un-Normalize to [0, 1] before Transform
                    x_denorm = x * std_tensor + mean_tensor 
                    
                    for _ in range(self.tta_rounds):
                        augmented_x = self.tta_transform(x_denorm)
                        x_final = self.norm(augmented_x) # Normalize again

                        aug_logits = model(x_final)
                        model_probs += torch.softmax(aug_logits, dim=1)
                
                # Calc mean (1 og + N TTA image)
                model_probs /= (self.tta_rounds + 1)
                final_probs += model_probs
                
        # calc mean on the number of ensembles
        final_probs /= len(self.models)
        return final_probs


In [21]:
def measure_inference_speed(
    model_pipeline: Union[nn.Module, HeavyTeacherPipeline], 
    model_name: str, 
    val_loader: DataLoader,
    device: str = DEVICE, 
    input_size: Tuple[int, int, int, int] = (1, 3, 224, 224)
) -> Tuple[float, float, float]:
    """Measures the inference latency and throughput (FPS) of a given model or pipeline.

    This profiling function includes a hardware warm-up sequence to initialize GPU clocks 
    and relies on `torch.cuda.synchronize()` alongside `time.perf_counter()` to ensure 
    highly accurate, sub-millisecond timekeeping. It isolates the computational latency 
    from data loading overheads by exclusively processing a pre-loaded tensor on the GPU.

    Args:
        model_pipeline (Union[nn.Module, HeavyTeacherPipeline]): The PyTorch model or 
            the custom ensemble pipeline to be evaluated.
        model_name (str): A descriptive identifier for the model, used for logging outputs.       
        val_loader (DataLoader): A Val data loader on the target device. (MUST)
        device (str, optional): The target computation device ('cuda' or 'cpu'). 
        input_size (Tuple[int, int, int, int], optional): The spatial dimensions of the 
            dummy tensor used if `val_loader` is None. Defaults to (1, 3, 224, 224).
    Returns:
        Tuple[float, float, float]: A tuple containing:
            - Mean latency (in milliseconds).
            - Standard deviation latency (in milliseconds).
            - Throughput (in Frames Per Second - FPS).
    """
    warmup_images, _ = next(iter(val_loader))
    warmup_images = warmup_images.to(device)

    print(f"\n[{model_name}] Initiating Hardware warm-up sequence...")
    with torch.no_grad():
        for _ in range(20): 
            if isinstance(model_pipeline, HeavyTeacherPipeline):
                _ = model_pipeline.predict(warmup_images)
            else:
                _ = model_pipeline(warmup_images)
                
    print(f"[{model_name}] Executing official measurement over the entire fold...")
    per_image_timings = []
    
    with torch.no_grad():
        for inputs, _ in tqdm(val_loader, desc=f"Profiling {model_name}", leave=False):
            inputs = inputs.to(device)
            batch_size = inputs.size(0)
            
            # Start
            if device == 'cuda':
                torch.cuda.synchronize()
            start_time = time.perf_counter()
            
            # Forward pass
            if isinstance(model_pipeline, HeavyTeacherPipeline):
                _ = model_pipeline.predict(inputs)
            else:
                _ = model_pipeline(inputs)
                
            # Stop
            if device == 'cuda':
                torch.cuda.synchronize()
            end_time = time.perf_counter()
            
            # Calculate of each image in a batch (ms)
            batch_time_ms = (end_time - start_time) * 1000.0
            per_image_ms = batch_time_ms / batch_size
            
            # Save result of images
            per_image_timings.extend([per_image_ms] * batch_size)
            
    # Metrics
    mean_lat = float(np.mean(per_image_timings))
    std_lat = float(np.std(per_image_timings))
    fps = 1000.0 / mean_lat
    
    print(f"Latency:    {mean_lat:.2f} ms ± {std_lat:.2f} ms")
    print(f"Throughput: {fps:.2f} FPS")
    
    return mean_lat, std_lat, fps

In [22]:
def main_profiling() -> None:
    """Main execution block to instantiate models and run speed profiling."""
    print("Loading Real Data from Test Set...")
    # DataLoader 
    _,val_loader = get_data_loaders() 
  
    # Instantiate core architectures (No pre-trained weights needed for speed profiling)

    resnet50_model = build_resnet50().to(DEVICE).eval()
    vgg19_model = build_vgg19().to(DEVICE).eval()

    # EdgeneXt xxs
    edgenext_xxs_model = build_edgenext_xxs_model().to(DEVICE).eval()
    # MobileViT xxs
    mobilevit_xxs_model = build_mobilevit_xxs_model().to(DEVICE).eval()
    
    # Load weights
    # Example: /kaggle/input/datasets/hophamsailam/resnet50-kaggle-train-test-validation/ResNet50.pth
    resnet50_model.load_state_dict(torch.load(MODEL3_DIR, map_location=DEVICE, weights_only=True))
    vgg19_model.load_state_dict(torch.load(MODEL4_DIR, map_location=DEVICE, weights_only=True))

    # 2. Load weights Student
    ## Example: /kaggle/input/datasets/xvmhieu/kaggle-edgenextxxs-ben-113/student_distilled.pth
    edgenext_xxs_model.load_state_dict(torch.load(MODEL5_DIR, map_location=DEVICE, weights_only=True))
    
    mobilevit_xxs_model.load_state_dict(torch.load(MODEL6_DIR, map_location=DEVICE, weights_only=True))
    
    # Construct the Heavy Teacher Pipeline
    teacher_pipeline = HeavyTeacherPipeline(
        models_list=[resnet50_model, vgg19_model],
        tta_rounds=5,
        device=DEVICE
    )
    # Measure Loader
    lat_t, std_t, fps_t = measure_inference_speed(teacher_pipeline, "Heavy Teacher", val_loader, DEVICE)
    lat_res, std_res, fps_res = measure_inference_speed(resnet50_model, MODEL3_NAME, val_loader, DEVICE)
    lat_vgg19, std_vgg19, fps_vgg19 = measure_inference_speed(vgg19_model, MODEL4_NAME, val_loader, DEVICE)
    lat_ed, std_ed, fps_ed = measure_inference_speed(edgenext_xxs_model, "EdgeNeXt-XXS", val_loader, DEVICE)
    lat_mvit, std_mvit, fps_mvit = measure_inference_speed(mobilevit_xxs_model, "MobileViT-XXS", val_loader, DEVICE)

    results = [{
        'Dataset': 'Kaggle Test Set',
        'Teacher': f"{lat_t:.2f} ± {std_t:.2f} ms | {fps_t:.2f} FPS",
        'ResNet50': f"{lat_res:.2f} ± {std_res:.2f} ms | {fps_res:.2f} FPS",
        'VGG19': f"{lat_vgg19:.2f} ± {std_vgg19:.2f} ms | {fps_vgg19:.2f} FPS",
        'EdgeNeXt-XXS': f"{lat_ed:.2f} ± {std_ed:.2f} ms | {fps_ed:.2f} FPS",
        'MobileViT-XXS': f"{lat_mvit:.2f} ± {std_mvit:.2f} ms | {fps_mvit:.2f} FPS"
    }]
    
    print("\n" + "*" * 100)
    print("FINAL INFERENCE SPEED SUMMARY")
    print("*" * 100)
    
    df_results = pd.DataFrame(results)
    
    
    latex_table = df_results.to_latex(
        index=False, 
        escape=False, 
        column_format='lccccc', 
        caption="Inference latency and throughput evaluated on Kaggle test dataset.",
        label="tab:inference_speed_kaggle"
    )
    print(latex_table)

In [23]:
main_profiling()

Loading Real Data from Test Set...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 176MB/s]


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 179MB/s]


Initializing EdgeNeXt-XXS student model...


...

Latency:    381.59 ms ± 5.65 ms
Throughput: 2.62 FPS



[EdgeNeXt-XXS] Initiating Hardware warm-up sequence...
[EdgeNeXt-XXS] Executing official measurement over the entire fold...


Latency:    11.22 ms ± 3.02 ms
Throughput: 89.16 FPS



[MobileViT-XXS] Initiating Hardware warm-up sequence...
[MobileViT-XXS] Executing official measurement over the entire fold...


Latency:    19.35 ms ± 4.38 ms
Throughput: 51.68 FPS

****************************************************************************************************
FINAL INFERENCE SPEED SUMMARY
****************************************************************************************************
\begin{table}
\caption{Inference latency and throughput evaluated on Kaggle test dataset.}
\label{tab:inference_speed_kaggle}
\begin{tabular}{lccccc}
\toprule
Dataset & Teacher & ResNet50 & VGG19 & EdgeNeXt-XXS & MobileViT-XXS \\
\midrule
Kaggle Test Set & 3258.81 ± 33.85 ms | 0.31 FPS & 123.07 ± 24.53 ms | 8.13 FPS & 381.59 ± 5.65 ms | 2.62 FPS & 11.22 ± 3.02 ms | 89.16 FPS & 19.35 ± 4.38 ms | 51.68 FPS \\
\bottomrule
\end{tabular}
\end{table}

